# Lab 5.2 – Kundensegmente mit Clustering

**Starter.** Kopie in `project/playground/` bearbeiten. Zellen der Reihe nach mit **Umschalt+Enter** ausführen.

Eingabe: `kunden_rfm.csv` aus dem Beispiel von Lab 2.3 (400 Kunden, Stichtag 30.09.2026). Es gibt keine Zielgröße: k-Means sucht Gruppen ähnlicher Kunden selbst. Ergebnis: `kunden_segmente.csv` und `segmente.html` in `ORDNER`.

In [ ]:
import pandas as pd
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

EINGABE = "project/examples/m02-data-preparation/l03-features/kunden_rfm.csv"
ORDNER = "project/playground/"     # hier landen kunden_segmente.csv und segmente.html

kunden = pd.read_csv(EINGABE)
MERKMALE = ["tage_seit_kauf", "haeufigkeit", "umsatz"]
print(len(kunden), "Kunden")
kunden[MERKMALE].describe().round(1)

## Schritt 1 (fertig): Skalieren

Umsatz reicht bis über 300.000 €, Häufigkeit bis etwa 700 Buchungen. Ohne Skalieren würde k-Means fast nur nach Euro gruppieren. `StandardScaler` bringt jede Spalte auf Mittelwert 0 und Standardabweichung 1.

In [ ]:
skaliert = StandardScaler().fit_transform(kunden[MERKMALE])   # je Spalte: Mittelwert 0, Standardabweichung 1
pd.DataFrame(skaliert, columns=MERKMALE).describe().round(1)

## Schritt 2 (fertig): k-Means mit 4 Segmenten

`n_clusters=4` ist eine fachliche Wahl; `random_state=42` und `n_init=10` machen das Ergebnis bei jedem Lauf gleich. Danach werden die Segmente nach Ø Umsatz nummeriert: Segment 1 hat den höchsten Ø Umsatz.

In [ ]:
ANZAHL_SEGMENTE = 4
kmeans = KMeans(n_clusters=ANZAHL_SEGMENTE, random_state=42, n_init=10)
kunden["segment"] = kmeans.fit_predict(skaliert)          # Nummern 0 bis 3, Reihenfolge zufällig

# Segmente nach Ø Umsatz neu nummerieren: 1 = höchster Ø Umsatz
reihenfolge = kunden.groupby("segment")["umsatz"].mean().sort_values(ascending=False).index
kunden["segment"] = kunden["segment"].map({alt: neu for neu, alt in enumerate(reihenfolge, start=1)})
kunden["segment"].value_counts().sort_index()

## Schritt 3 (fertig): Kennzahlen und Streudiagramm je Segment

In [ ]:
segment_tabelle = kunden.groupby("segment").agg(
    kunden=("kunde_id", "count"),
    tage_seit_kauf=("tage_seit_kauf", "mean"),
    haeufigkeit=("haeufigkeit", "mean"),
    umsatz=("umsatz", "mean"),
    umsatz_summe=("umsatz", "sum"),
).round(0)
segment_tabelle["anteil_umsatz_prozent"] = (
    segment_tabelle["umsatz_summe"] / segment_tabelle["umsatz_summe"].sum() * 100
).round(1)
segment_tabelle

In [ ]:
fig = px.scatter(
    kunden, x="haeufigkeit", y="umsatz", color=kunden["segment"].astype(str),
    hover_data=["kunde_id", "tage_seit_kauf"],
    title="14 Top-Kunden bringen gut ein Viertel des Umsatzes",
    labels={"haeufigkeit": "Buchungen bis 30.09.2026", "umsatz": "Umsatz in €", "color": "Segment"},
    category_orders={"color": ["1", "2", "3", "4"]},
)
fig

## Schritt 4: Segmente benennen (Aufgabe 1)

Die Tabelle oben zeigt je Segment Anzahl, Ø Tage seit dem letzten Kauf, Ø Buchungen und Ø Umsatz. Die Zelle darunter mit dem vorgegebenen Prompt aus der Übung (Aufgabe 3) füllen, dann die Prüfzelle ausführen.

In [ ]:
# >>> AUFGABE 1: Spalte segment_name mit einem Namen je Segment ergänzen – Prompt siehe Übung, Aufgabe 3
kunden["segment_name"] = None

# <<< ENDE AUFGABE 1
kunden.groupby(["segment", "segment_name"], dropna=False).size()

In [ ]:
# Prüfzelle – nicht ändern
def pruefe_namen(t):
    if "segment_name" not in t.columns or t["segment_name"].isna().any():
        return "nicht OK – Spalte segment_name fehlt oder ist leer: jedem Segment 1–4 einen Namen geben."
    if (t.groupby("segment")["segment_name"].nunique() != 1).any():
        return "nicht OK – ein Segment hat mehrere Namen: je Segmentnummer genau ein Name."
    if t["segment_name"].nunique() != ANZAHL_SEGMENTE:
        return f"nicht OK – {t['segment_name'].nunique()} verschiedene Namen statt {ANZAHL_SEGMENTE}: jeden Namen nur einmal vergeben."
    return "OK – " + ", ".join(f"{s} = {n}" for s, n in t.groupby("segment")["segment_name"].first().items())


print(pruefe_namen(kunden))

## Ergebnis speichern

`kunden_segmente.csv` enthält je Kunde Segment und Name; die Segmenttabelle bekommt die Namen dazu.

In [ ]:
kunden.to_csv(ORDNER + "kunden_segmente.csv", index=False)
fig.write_html(ORDNER + "segmente.html", include_plotlyjs="cdn")
print("Gespeichert:", ORDNER + "kunden_segmente.csv", "und", ORDNER + "segmente.html")
segment_tabelle["segment_name"] = kunden.groupby("segment")["segment_name"].first()
segment_tabelle

## Erweiterung: Elbow-Diagramm

Für 1 bis 8 Segmente misst `inertia_`, wie weit die Kunden im Schnitt von ihrem Segment-Mittelpunkt entfernt liegen (kleiner = enger). Wo die Kurve abknickt („Ellbogen“), bringt ein weiteres Segment wenig. Danach `ANZAHL_SEGMENTE` in Schritt 2 auf 3 oder 5 setzen und ab Schritt 2 neu ausführen.

In [ ]:
werte = []
for k in range(1, 9):
    modell = KMeans(n_clusters=k, random_state=42, n_init=10).fit(skaliert)
    werte.append({"segmente": k, "abstand_innen": round(modell.inertia_, 1)})
elbow = pd.DataFrame(werte)
px.line(elbow, x="segmente", y="abstand_innen", markers=True,
        title="Elbow: nach 3–4 Segmenten flacht die Kurve ab",
        labels={"segmente": "Anzahl Segmente", "abstand_innen": "Abstand innerhalb der Segmente (inertia_)"})